# 📚 03. 중요 — 실전 활용을 위한 Python

이 노트북은 **pandas를 실제 업무에서 활용하기 위해** 반드시 알아야 할 개념을 다룹니다.

## 학습 목표
| 개념 | pandas 연결 포인트 |
|------|-------------------|
| 클래스 기초 | DataFrame/Series가 클래스 인스턴스임을 이해 |
| 파일 입출력 | `pd.read_csv()`, `df.to_csv()`, `read_excel()` |
| 예외 처리 | 결측값·타입 오류·파일 없음 등 실전 오류 방어 |

> **전제**: 01_필수, 02_핵심 노트북을 먼저 완료해주세요.

In [ ]:
# =============================================
# 공통 데이터 준비
# =============================================

import pandas as pd
import os  # 파일 처리에 사용

# 설명: 이 노트북 전체에서 사용할 판매 데이터
판매_딕셔너리 = {
    '상품명':   ['노트북', '마우스', '키보드', '모니터', '헤드셋', '웹캠'],
    '카테고리': ['IT기기', 'IT기기', 'IT기기', 'IT기기', '음향기기', '영상기기'],
    '가격':     [1200000, 35000, 75000, 450000, 89000, 120000],
    '수량':     [5, 50, 30, 8, 20, 12],
    '지역':     ['서울', '부산', '서울', '대구', '서울', '인천'],
    '평점':     [4.8, 4.5, 4.7, 4.6, 4.3, 4.9],
}

df = pd.DataFrame(판매_딕셔너리)
df['매출'] = df['가격'] * df['수량']

print('데이터 준비 완료!')
print(df)

---
## 1장. 클래스 기초

클래스는 **데이터(속성)**와 **기능(메서드)**을 하나로 묶는 설계도입니다.  
클래스로 만든 **인스턴스(객체)**가 실제로 사용하는 것입니다.

> **pandas 연결**:  
> `pd.DataFrame`은 클래스, `df = pd.DataFrame(...)` 하면 인스턴스 생성  
> `df.shape`, `df.columns`, `df.dtypes` → DataFrame 클래스의 **속성(attribute)**  
> `df.head()`, `df.describe()`, `df.groupby()` → DataFrame 클래스의 **메서드(method)**

In [ ]:
# =============================================
# 1-1. 클래스 기본 구조
# =============================================

# 설명: 클래스 = 설계도, 인스턴스 = 실제 사용하는 객체
# pandas 연결: DataFrame 클래스가 이런 구조로 만들어져 있음

class 상품:
    """상품 정보를 담는 클래스"""
    
    # __init__: 인스턴스 생성 시 자동 호출되는 초기화 메서드
    # self: 인스턴스 자기 자신을 가리키는 참조
    def __init__(self, 이름, 가격, 수량):
        # 속성(attribute): 인스턴스가 가지는 데이터
        self.이름 = 이름
        self.가격 = 가격
        self.수량 = 수량
        self.매출 = 가격 * 수량  # 생성 시 자동 계산
    
    # 메서드(method): 인스턴스가 가지는 기능
    def 정보_출력(self):
        """상품 정보를 출력하는 메서드"""
        print(f'상품명: {self.이름}')
        print(f'가격:   {self.가격:,}원')
        print(f'수량:   {self.수량}개')
        print(f'매출:   {self.매출:,}원')
    
    def 할인_적용(self, 할인율=0.1):
        """할인율을 적용한 가격을 반환"""
        return int(self.가격 * (1 - 할인율))

# 인스턴스 생성
노트북 = 상품('노트북', 1200000, 5)
마우스 = 상품('마우스', 35000, 50)

print('=== 노트북 정보 ===')
노트북.정보_출력()

print(f'\n10% 할인가: {노트북.할인_적용():,}원')
print(f'20% 할인가: {노트북.할인_적용(할인율=0.2):,}원')

In [ ]:
# =============================================
# 1-2. DataFrame은 클래스다!
# =============================================

# 설명: DataFrame 클래스의 속성과 메서드를 직접 확인
# pandas 연결: 클래스를 이해하면 pandas 문서를 읽는 것이 쉬워짐

print('=== DataFrame 클래스 확인 ===')
print(f'타입: {type(df)}')          # → <class 'pandas.core.frame.DataFrame'>
print(f'클래스 이름: {type(df).__name__}')  # → DataFrame

print('\n=== DataFrame의 속성(attribute) - 괄호 없음 ===')
print(f'shape (행, 열): {df.shape}')         # → (6, 7)
print(f'columns: {list(df.columns)}')        # → 컬럼명 리스트
print(f'index: {list(df.index)}')            # → 인덱스 리스트
print(f'dtypes:\n{df.dtypes}')               # → 각 컬럼의 타입

print('\n=== DataFrame의 메서드(method) - 괄호 있음 ===')
print(df.head(3))                            # 처음 3행
print(df.describe())                         # 기술통계

In [ ]:
# =============================================
# 1-3. 판매 분석기 클래스 만들기
# =============================================

# 설명: DataFrame을 내부에 담아 분석 기능을 제공하는 클래스
# pandas 연결: pandas 메서드 체이닝과 유사한 구조

class 판매_분석기:
    """판매 데이터를 분석하는 클래스"""
    
    def __init__(self, 데이터프레임):
        # 설명: 외부 DataFrame을 받아 내부에 저장
        self.df = 데이터프레임.copy()  # copy()로 원본 보호
        print(f'분석기 초기화 완료 (데이터: {len(self.df)}행)')
    
    def 기본_통계(self):
        """기본 통계 정보 출력"""
        print('=== 기본 통계 ===')
        print(f'  총 상품 수: {len(self.df)}개')
        print(f'  총 매출: {self.df["매출"].sum():,}원')
        print(f'  평균 가격: {self.df["가격"].mean():,.0f}원')
        print(f'  최고 평점: {self.df["평점"].max()}')
    
    def 카테고리별_분석(self):
        """카테고리별 매출 합계 반환"""
        return self.df.groupby('카테고리')['매출'].sum().sort_values(ascending=False)
    
    def 상위_상품(self, n=3, 기준='매출'):
        """매출/평점 등 기준으로 상위 n개 상품 반환"""
        return self.df.nlargest(n, 기준)[['상품명', 기준]]

# 인스턴스 생성
분석기 = 판매_분석기(df)

# 메서드 호출
분석기.기본_통계()

print('\n=== 카테고리별 매출 ===')
print(분석기.카테고리별_분석())

print('\n=== 매출 TOP 3 ===')
print(분석기.상위_상품(n=3, 기준='매출'))

print('\n=== 평점 TOP 3 ===')
print(분석기.상위_상품(n=3, 기준='평점'))

In [ ]:
# =============================================
# 1-4. 클래스 상속: 기능 확장
# =============================================

# 설명: 기존 클래스를 상속받아 기능을 추가하는 방법
# pandas 연결: pandas 내부도 상속으로 클래스 계층이 구성됨
#              NDFrame → DataFrame / Series

class 고급_분석기(판매_분석기):  # 판매_분석기를 상속
    """판매 분석기를 확장한 고급 분석기"""
    
    def __init__(self, 데이터프레임, 분석자명='미지정'):
        super().__init__(데이터프레임)  # 부모 클래스 초기화
        self.분석자명 = 분석자명
    
    def 종합_리포트(self):
        """전체 분석 리포트 출력"""
        print(f'=== 종합 리포트 (분석자: {self.분석자명}) ===')
        self.기본_통계()  # 부모의 메서드 호출!
        print('\n카테고리별 매출:')
        print(self.카테고리별_분석())

# 사용
고급분석기 = 고급_분석기(df, 분석자명='홍길동')
고급분석기.종합_리포트()

---
## 2장. 파일 입출력

실제 데이터 분석은 파일에서 데이터를 읽고, 결과를 파일에 저장합니다.  
Python의 기본 파일 처리를 이해하면 pandas의 `read_csv()`, `to_csv()`가 쉬워집니다.

> **pandas 연결**:  
> `pd.read_csv('파일명.csv')` → CSV 파일을 DataFrame으로 읽기  
> `df.to_csv('파일명.csv', index=False)` → DataFrame을 CSV로 저장

In [ ]:
# =============================================
# 2-1. Python 기본 파일 쓰기
# =============================================

# 설명: open()으로 파일을 열고 write()로 내용 작성
# pandas 연결: to_csv() 내부도 이와 같은 파일 쓰기를 사용

# with 문: 파일을 자동으로 닫아주는 안전한 방법
# 주의: with 블록이 끝나면 자동으로 close() 호출 → 파일 안전하게 닫힘
with open('test_output.txt', 'w', encoding='utf-8') as 파일:
    파일.write('안녕하세요!\n')           # \n은 줄바꿈
    파일.write('파일 쓰기 테스트입니다.\n')
    파일.write(f'상품 수: {len(df)}개\n')

print('test_output.txt 파일 생성 완료!')

# 파일이 생성되었는지 확인
print(f'파일 존재 여부: {os.path.exists("test_output.txt")}')
print(f'파일 크기: {os.path.getsize("test_output.txt")} bytes')

In [ ]:
# =============================================
# 2-2. Python 기본 파일 읽기
# =============================================

# 설명: open()으로 파일을 열고 read()로 내용 읽기
# 모드: 'r' = 읽기, 'w' = 쓰기, 'a' = 추가, 'rb' = 바이너리 읽기

# 전체 파일 읽기
with open('test_output.txt', 'r', encoding='utf-8') as 파일:
    내용 = 파일.read()
print('=== 전체 읽기 ===')
print(내용)

# 한 줄씩 읽기
with open('test_output.txt', 'r', encoding='utf-8') as 파일:
    줄들 = 파일.readlines()  # 리스트로 반환
print('=== 줄 단위 읽기 ===')
for i, 줄 in enumerate(줄들, 1):
    print(f'  {i}번째 줄: {줄.strip()}')  # strip()으로 앞뒤 공백/줄바꿈 제거

In [ ]:
# =============================================
# 2-3. CSV 파일 직접 만들기 (pandas 없이)
# =============================================

# 설명: CSV = Comma Separated Values (쉼표로 구분된 값)
#       pandas의 to_csv()가 하는 일을 직접 구현해봄

# CSV 구조:
# 헤더줄: 상품명,카테고리,가격,수량
# 데이터줄: 노트북,IT기기,1200000,5

상품명들   = ['노트북', '마우스', '키보드', '모니터', '헤드셋', '웹캠']
카테고리들 = ['IT기기', 'IT기기', 'IT기기', 'IT기기', '음향기기', '영상기기']
가격들     = [1200000, 35000, 75000, 450000, 89000, 120000]
수량들     = [5, 50, 30, 8, 20, 12]

# CSV 파일 직접 작성
with open('판매_데이터.csv', 'w', encoding='utf-8-sig') as 파일:
    # utf-8-sig: Excel에서 한글 CSV 열 때 BOM 필요
    
    # 헤더 줄 작성
    파일.write('상품명,카테고리,가격,수량,매출\n')
    
    # 데이터 줄 작성
    for i in range(len(상품명들)):
        매출 = 가격들[i] * 수량들[i]
        줄 = f'{상품명들[i]},{카테고리들[i]},{가격들[i]},{수량들[i]},{매출}\n'
        파일.write(줄)

print('판매_데이터.csv 생성 완료!')

# 확인: 직접 읽어보기
with open('판매_데이터.csv', 'r', encoding='utf-8-sig') as 파일:
    print(파일.read())

In [ ]:
# =============================================
# 2-4. pandas로 CSV 읽고 쓰기
# =============================================

# 설명: pandas가 제공하는 강력한 CSV 입출력 함수
# pandas 연결: 이것이 실제 업무에서 가장 자주 쓰는 기능!

# DataFrame → CSV 저장
df.to_csv('판매_pandas.csv', index=False, encoding='utf-8-sig')
# index=False: 인덱스(0,1,2...) 컬럼 제외하고 저장
print('판매_pandas.csv 저장 완료!')

# CSV → DataFrame 읽기
df_읽기 = pd.read_csv('판매_pandas.csv', encoding='utf-8-sig')
print('\n=== CSV에서 읽은 DataFrame ===')
print(df_읽기)

print('\n=== 읽기 옵션들 ===')
# 특정 컬럼만 읽기
df_일부 = pd.read_csv('판매_pandas.csv', encoding='utf-8-sig',
                       usecols=['상품명', '가격', '매출'])
print('일부 컬럼만:')
print(df_일부.head(3))

In [ ]:
# =============================================
# 2-5. 다양한 파일 형식
# =============================================

# 설명: pandas는 CSV 외에도 다양한 형식을 지원
# 주의: Excel 읽기는 openpyxl 패키지 필요 (pip install openpyxl)

print('=== 판다스 지원 파일 형식 ===')
형식_안내 = [
    ('CSV',    'pd.read_csv("파일.csv")',       'df.to_csv("파일.csv")'),
    ('Excel',  'pd.read_excel("파일.xlsx")',    'df.to_excel("파일.xlsx")'),
    ('JSON',   'pd.read_json("파일.json")',     'df.to_json("파일.json")'),
    ('Parquet','pd.read_parquet("파일.parq")', 'df.to_parquet("파일.parq")'),
    ('SQL DB', 'pd.read_sql(쿼리, 연결)',       'df.to_sql(테이블명, 연결)'),
]

for 형식, 읽기, 쓰기 in 형식_안내:
    print(f'  [{형식}]')
    print(f'    읽기: {읽기}')
    print(f'    쓰기: {쓰기}')

# JSON 저장/읽기 실습
df.to_json('판매_데이터.json', orient='records', force_ascii=False, indent=2)
print('\n판매_데이터.json 저장 완료!')

df_json = pd.read_json('판매_데이터.json')
print('JSON에서 읽은 데이터:')
print(df_json[['상품명', '가격']].head(3))

---
## 3장. 예외 처리 (try / except)

예외 처리는 **오류가 발생해도 프로그램이 멈추지 않도록** 보호합니다.  
실제 데이터는 항상 예상치 못한 형태로 들어오기 때문에 필수적입니다.

> **pandas 연결**:  
> 결측값(NaN) 처리, 타입 변환 오류, 파일 없음 오류 등을 방어

In [ ]:
# =============================================
# 3-1. 기본 try / except 구조
# =============================================

# 설명: try 블록에서 오류가 발생하면 except 블록이 실행됨
# 구조:
#   try: 실행할 코드
#   except 오류종류 as e: 오류 처리
#   else: 오류 없을 때 실행
#   finally: 오류 유무 상관없이 항상 실행

# 예시 1: ZeroDivisionError (0으로 나누기)
print('=== ZeroDivisionError 처리 ===')
try:
    결과 = 100 / 0  # 오류 발생!
    print(f'결과: {결과}')  # 실행되지 않음
except ZeroDivisionError as e:
    print(f'오류 발생: {e}')      # → division by zero
    print('0으로 나눌 수 없습니다!')

# 예시 2: ValueError (잘못된 타입 변환)
print('\n=== ValueError 처리 ===')
입력값들 = ['100', '200', '삼백', '400', 'N/A']
변환된_숫자들 = []

for 값 in 입력값들:
    try:
        숫자 = int(값)              # 문자 '삼백', 'N/A'는 오류 발생
        변환된_숫자들.append(숫자)
    except ValueError:
        print(f'  "{값}"은 숫자로 변환 불가 → 0으로 대체')
        변환된_숫자들.append(0)    # 오류 시 0으로 대체

print('변환 결과:', 변환된_숫자들)

In [ ]:
# =============================================
# 3-2. 파일 읽기 예외 처리
# =============================================

# 설명: 파일이 없거나 형식이 잘못되었을 때 오류 처리
# pandas 연결: pd.read_csv()가 실패할 때 대비

def 안전한_CSV_읽기(파일경로):
    """CSV 파일을 안전하게 읽는 함수. 오류 시 빈 DataFrame 반환"""
    try:
        df = pd.read_csv(파일경로, encoding='utf-8-sig')
        print(f'✅ 파일 읽기 성공: {파일경로} ({len(df)}행)')
        return df
    except FileNotFoundError:
        # 오류: 파일이 존재하지 않음
        print(f'❌ 파일 없음: {파일경로}')
        return pd.DataFrame()  # 빈 DataFrame 반환
    except pd.errors.ParserError:
        # 오류: CSV 형식이 잘못됨 (파싱 실패)
        print(f'❌ CSV 형식 오류: {파일경로}')
        return pd.DataFrame()
    except Exception as e:
        # 예상치 못한 오류는 무엇이든 처리
        print(f'❌ 알 수 없는 오류: {e}')
        return pd.DataFrame()
    finally:
        # finally: 성공이든 실패든 항상 실행
        print('   (파일 읽기 시도 완료)')

# 정상 파일 읽기
df1 = 안전한_CSV_읽기('판매_pandas.csv')

print()

# 없는 파일 읽기
df2 = 안전한_CSV_읽기('없는파일.csv')
print(f'빈 DataFrame: {df2.empty}')  # → True

In [ ]:
# =============================================
# 3-3. 결측값(NaN) 처리
# =============================================

# 설명: 실제 데이터에는 빈 값(NaN)이 자주 있음
# pandas 연결: NaN이 포함된 연산 결과 → NaN이 전파됨
#              이를 처리하는 방법: fillna(), dropna(), interpolate()

import numpy as np

# 결측값이 포함된 데이터 생성
df_결측 = pd.DataFrame({
    '상품명': ['노트북', '마우스', '키보드', '모니터', '헤드셋'],
    '가격':   [1200000, None, 75000, 450000, None],   # None → NaN
    '수량':   [5, 50, None, 8, 20],
    '평점':   [4.8, 4.5, 4.7, None, 4.3],
})

print('=== 결측값 포함 데이터 ===')
print(df_결측)

print('\n=== 결측값 확인 ===')
print(df_결측.isnull())

print('\n=== 컬럼별 결측값 개수 ===')
print(df_결측.isnull().sum())

In [ ]:
# =============================================
# 3-4. 결측값 처리 전략
# =============================================

# 설명: 결측값을 어떻게 처리할지 전략 선택
# pandas 연결: fillna(), dropna(), interpolate()

# 전략 1: fillna() - 특정 값으로 채우기
df_채움 = df_결측.copy()
df_채움['가격'] = df_채움['가격'].fillna(df_결측['가격'].mean())  # 평균으로 채우기
df_채움['수량'] = df_채움['수량'].fillna(0)                       # 0으로 채우기
df_채움['평점'] = df_채움['평점'].fillna(method='ffill')          # 앞 값으로 채우기

print('=== fillna() 처리 결과 ===')
print(df_채움)

# 전략 2: dropna() - 결측값 있는 행 제거
df_제거 = df_결측.dropna()  # 결측값 있는 행 모두 제거
print('\n=== dropna() 처리 결과 ===')
print(df_제거)
print(f'제거 후 행 수: {len(df_제거)} (원본: {len(df_결측)})')

In [ ]:
# =============================================
# 3-5. 타입 변환 예외 처리
# =============================================

# 설명: 실제 CSV 데이터는 문자열로 들어오는 경우가 많음
# 이를 숫자로 변환할 때 오류가 발생할 수 있음
# pandas 연결: pd.to_numeric(errors='coerce') 로 안전하게 변환

# 오류 데이터 시뮬레이션 (CSV에서 읽어온 상황)
df_문자열 = pd.DataFrame({
    '상품명': ['노트북', '마우스', '키보드', '모니터'],
    '가격':   ['1,200,000', '35000', 'N/A', '450,000'],  # 콤마, N/A 포함
    '수량':   ['5', '오십', '30', '8'],                   # 한글 숫자 포함
})

print('=== 문자열 데이터 ===')
print(df_문자열)
print('데이터 타입:', df_문자열.dtypes.to_dict())

# 방법 1: 정규식으로 콤마 제거 후 변환
df_변환 = df_문자열.copy()
df_변환['가격_정제'] = df_문자열['가격'].str.replace(',', '')  # 콤마 제거

# 방법 2: pd.to_numeric(errors='coerce') - 오류 시 NaN으로
df_변환['가격_숫자'] = pd.to_numeric(df_변환['가격_정제'], errors='coerce')
df_변환['수량_숫자'] = pd.to_numeric(df_문자열['수량'],    errors='coerce')

print('\n=== 숫자 변환 결과 ===')
print(df_변환[['상품명', '가격', '가격_숫자', '수량', '수량_숫자']])

---
## 4장. 종합 실습

In [ ]:
# =============================================
# 4-1. 종합 실습: 클래스 + 파일 I/O + 예외 처리
# =============================================

# 설명: 세 가지 개념을 통합한 실전적인 데이터 파이프라인

class 데이터_파이프라인:
    """CSV 파일을 읽어 분석하고 결과를 저장하는 파이프라인"""
    
    def __init__(self, 입력경로, 출력경로):
        self.입력경로 = 입력경로
        self.출력경로 = 출력경로
        self.df = None  # 데이터 아직 없음
    
    def 데이터_읽기(self):
        """CSV 파일을 읽어 self.df에 저장"""
        try:
            self.df = pd.read_csv(self.입력경로, encoding='utf-8-sig')
            print(f'✅ 읽기 성공: {len(self.df)}행 {len(self.df.columns)}열')
        except FileNotFoundError:
            print(f'❌ 파일 없음: {self.입력경로}')
            self.df = pd.DataFrame()
        return self  # 메서드 체이닝 지원 (self 반환)
    
    def 데이터_검증(self):
        """데이터 품질 검증"""
        if self.df is None or self.df.empty:
            print('❌ 검증 불가: 데이터 없음')
            return self
        
        결측_수 = self.df.isnull().sum().sum()
        print(f'검증: 결측값 {결측_수}개 발견')
        
        if 결측_수 > 0:
            # 숫자형 컬럼은 평균으로 채우기
            for col in self.df.select_dtypes(include='number').columns:
                self.df[col] = self.df[col].fillna(self.df[col].mean())
            print(f'✅ 결측값 평균으로 대체 완료')
        
        return self  # 메서드 체이닝 지원
    
    def 분석_실행(self):
        """기본 분석 수행"""
        if self.df is None or self.df.empty:
            return self
        
        try:
            print('\n=== 분석 결과 ===')
            print(self.df.describe().round(0))
        except Exception as e:
            print(f'❌ 분석 오류: {e}')
        
        return self
    
    def 결과_저장(self):
        """처리된 데이터를 CSV로 저장"""
        if self.df is None or self.df.empty:
            print('❌ 저장할 데이터 없음')
            return self
        
        try:
            self.df.to_csv(self.출력경로, index=False, encoding='utf-8-sig')
            print(f'\n✅ 결과 저장: {self.출력경로}')
        except PermissionError:
            print(f'❌ 권한 없음: {self.출력경로} 파일이 열려있지 않은지 확인')
        
        return self


# 파이프라인 실행 (메서드 체이닝)
파이프라인 = 데이터_파이프라인('판매_pandas.csv', '분석_결과.csv')
(
    파이프라인
    .데이터_읽기()
    .데이터_검증()
    .분석_실행()
    .결과_저장()
)

In [ ]:
# =============================================
# 마무리: 생성된 파일 정리
# =============================================

# 설명: 실습 중 생성된 파일 목록 확인
생성된_파일들 = [
    'test_output.txt',
    '판매_데이터.csv',
    '판매_pandas.csv',
    '판매_데이터.json',
    '분석_결과.csv',
]

print('=== 생성된 파일 목록 ===')
for 파일명 in 생성된_파일들:
    if os.path.exists(파일명):
        크기 = os.path.getsize(파일명)
        print(f'  ✅ {파일명} ({크기} bytes)')
    else:
        print(f'  ❌ {파일명} (없음)')

---
## 정리

| 개념 | 핵심 내용 | pandas 연결 |
|------|-----------|-------------|
| **클래스** | `class`, `__init__`, `self`, 메서드 | DataFrame/Series가 클래스 인스턴스 |
| **상속** | `class 자식(부모)`, `super().__init__()` | NDFrame → DataFrame 상속 구조 |
| **파일 I/O** | `open()`, `with`, `read/write` | `pd.read_csv()`, `df.to_csv()` |
| **예외 처리** | `try/except/finally` | 결측값·파일 오류·타입 오류 방어 |
| **pd.to_numeric** | `errors='coerce'` → NaN | 문자 섞인 숫자 컬럼 안전 변환 |

### 다음 단계
**04_심화.ipynb** → 이터레이터, `*args/**kwargs`, 데코레이터를 배웁니다.  
이 개념들은 pandas의 고급 기능(`pipe()`, `itertuples()`)과 직결됩니다.